# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- DOI: 10.71728/senscience.y7m0-f273
- URL: https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json


In [ ]:
# Make sure `mlcroissant` is installed in the Jupyter environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

dataset = mlc.Dataset(croissant_url)
# Access metadata object
metadata = dataset.metadata

# Print title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore the available record sets, fields, and their `@id` values. This helps to understand the data structure and how to reference entities for extraction and analysis.

Let's print all available record sets, their IDs, and inspect the fields in each.

In [ ]:
# List all record sets in the dataset

record_sets = list(dataset.record_sets())  # Each is a RecordSetMetadata object
print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs.id} | name: {rs.name} | description: {getattr(rs, 'description', '')}")

if record_sets:
    print("\nFields in each Record Set:")
    for rs in record_sets:
        print(f"\nRecord Set: {rs.id} ({rs.name})")
        fields = list(rs.fields)
        for field in fields:
            # Each field is a FieldMetadata object
            print(f"    - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")

    # For example, to see example records in the first record set (if present):
    print(f"\nExample records from record set: {record_sets[0].id}")
    for i, record in enumerate(dataset.records(record_set=record_sets[0].id)):
        print(record)
        if i > 2:
            break


## 3. Data Extraction
Load data from each available record set into a pandas DataFrame using the record set `@id`.
All subsequent operations reference the record set and field/column entities by their `@id` fields for clarity and reproducibility.

In [ ]:
# Extract data from each record set into a DataFrame
dataframes = {}

# Gather all record set IDs
record_set_ids = [rs.id for rs in record_sets]
print(f"\nRecord Set IDs: {record_set_ids}")

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from record set {record_set_id}")

# Show the columns (field @ids) of the first record set, if any
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns (field @id) in record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's perform standard EDA operations:
- Filtering records based on a numeric field
- Normalizing a numeric field
- Grouping (aggregating) by a key attribute

All field references use their record set and field `@id`s.

In [ ]:
# Example: Choose a record set and numeric field
# (Update numerics as appropriate for dataset fields discovered above)

if record_set_ids:
    # Choose the first record set as demonstration
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    
    print(f"Columns in {record_set_id}:", df.columns.tolist())
    
    # Guess numeric field: find float or int dtype if possible
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id is not None:
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 10
        # Filter
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered to {len(filtered_df)} records with {numeric_field_id} > {threshold}")

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical column, if present
        # Find first non-numeric column
        group_field_id = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id is not None:
            # Group and compute mean
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print(f"No numeric field found in record set {record_set_id}.")

## 5. Visualization
Visualize the distribution of a numeric field, or a relationship if applicable, using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id is not None:
    # Distribution plot for normalized numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=20, kde=True)
    plt.title(f'Normalized Distribution of {numeric_field_id} in {record_set_id}')
    plt.xlabel(f"{numeric_field_id} (normalized)")
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group, if group field exists
    if group_field_id is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id} (Filtered)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, you used the FAIR² Croissant dataset schema to:
- Load metadata and records using `mlcroissant`
- Inspect and reference record sets and fields by their `@id`s
- Extract and analyze dataset contents using standard pandas techniques
- Visualize key numeric fields

This approach is extensible to other Croissant datasets. For comprehensive analysis, consult documentation for field semantics and continue with statistical or modeling work tailored to the data's scientific context.